In [1]:
from agent.graph import get_yolov11_detector, get_owlv2_detector, upload_detection_image_to_r2
import httpx
from IPython.display import Image, display

2026-03-08 17:08:47.818170316 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
IMAGE_URL = "https://ttymsbsmurxtpsrvlokw.supabase.co/storage/v1/object/public/thesis-bucket/uploads/images/a8rgqu70ixa.JPG"
CONFIDENCE_THRESHOLD = 0.3

async def run_closed_set_leaf_detection(image_url: str, confidence_threshold: float = 0.3):
    async with httpx.AsyncClient() as client:
        response = await client.get(image_url)
        response.raise_for_status()
        image_binary = response.content

    detector = get_yolov11_detector()
    results = await detector.predict(
        image_input=image_binary,
        conf_threshold=confidence_threshold,
    )

    viz_bytes = detector.visualize_detections(
        image_input=image_binary,
        detections=results,
        output_format='bytes'
    )

    viz_url = upload_detection_image_to_r2(viz_bytes)

    return results, viz_url

results, viz_url = await run_closed_set_leaf_detection(IMAGE_URL, CONFIDENCE_THRESHOLD)

print(f"Ditemukan {len(results)} deteksi\n")
print("=" * 60)
print(f"URL Visualisasi: {viz_url}\n")
display(Image(url=viz_url))

for i, det in enumerate(results, 1):
    box = det["box"]
    print(f"Deteksi {i}: {det['label']} | skor: {det['score']:.3f} | kotak: [{box[0]:.1f}, {box[1]:.1f}, {box[2]:.1f}, {box[3]:.1f}]")

YOLOv11Detector initialized, loading YOLOv11 Small model from: /root/workspace/repos/thesis/research/react-agent-engine/models/yolov11
✓ Loaded YOLOv11 Small model: yolo11s.sim.onnx
Ditemukan 1 deteksi

URL Visualisasi: https://thesis-assets.andyathsid.com/detection-results/detection_95b018b0-9bd6-4a78-af9d-0b8063b0e168.png



Deteksi 1: 0 | skor: 0.874 | kotak: [94.0, 55.3, 500.0, 333.1]


In [3]:
OWL_IMAGE_URL = "https://ttymsbsmurxtpsrvlokw.supabase.co/storage/v1/object/public/thesis-bucket/plantwild/evaluation/images/0002_apple_black_rot.jpg"
OWL_LABELS = ["fruit", "apple"]
OWL_THRESHOLD = 0.7

async def run_open_vocabulary_detection(image_url: str, labels: list[str], threshold: float = 0.3):
    async with httpx.AsyncClient() as client:
        response = await client.get(image_url)
        response.raise_for_status()
        image_binary = response.content

    detector = get_owlv2_detector()
    results = await detector.predict(
        image_input=image_binary,
        labels=labels,
        threshold=threshold,
    )

    viz_bytes = detector.visualize_detections(
        image_input=image_binary,
        detections=results,
        output_format='bytes'
    )

    viz_url = upload_detection_image_to_r2(viz_bytes)

    return results, viz_url

results_owl, viz_url_owl = await run_open_vocabulary_detection(OWL_IMAGE_URL, OWL_LABELS, OWL_THRESHOLD)

print(f"Ditemukan {len(results_owl)} deteksi kosakata terbuka\n")
print("=" * 60)
print(f"URL Visualisasi: {viz_url_owl}\n")
display(Image(url=viz_url_owl))

for i, det in enumerate(results_owl, 1):
    box = det["box"]
    print(f"Deteksi {i}: {det['label']} | skor: {det['score']:.3f} | kotak: [{box[0]:.1f}, {box[1]:.1f}, {box[2]:.1f}, {box[3]:.1f}]")

Ditemukan 1 deteksi kosakata terbuka

URL Visualisasi: https://thesis-assets.andyathsid.com/detection-results/detection_bbc6c60c-b2e0-4e5f-a5cf-b3be2e29d6a6.png



Deteksi 1: apple | skor: 0.777 | kotak: [358.5, 135.8, 813.2, 554.3]
